# Daily Challenge: Build a Tiny Agent with Tools — Teacher (Solution)
Author: arielzin33@gmail.com

A `smolagents` `ToolCallingAgent` with two tools — `KBLookupTool` and `MathTool` — answering 3 test queries, citing the knowledge base with `[kb:N]` tags when used.

**A real finding, from actually testing this before writing it:** the assignment's suggested default model (`sshleifer/tiny-gpt2`) and its fallback (`FakeListChatModel`) **both don't work** with current `smolagents`:

- `sshleifer/tiny-gpt2` has no chat template (it's a raw, non-chat GPT-2 checkpoint), and `TransformersModel` now requires one — it crashes immediately with `ValueError: Cannot use chat template functions because tokenizer.chat_template is not set`, confirmed by actually loading it.
- `FakeListChatModel` is a **LangChain** class, not a `smolagents` one — `smolagents` has no such stub at all (confirmed: `smolagents` exports nothing matching `*ake*`/`*stub*`/`*mock*`).
- Even a genuinely small **instruction-tuned** model, `HuggingFaceTB/SmolLM2-135M-Instruct` (chosen because it's designed for exactly this kind of local smolagents demo), was tested directly and **never produced one valid tool call across 4 attempts** — it just wrote freeform Python code instead of calling the registered tool, taking 25–35 seconds per step on CPU before hitting `max_steps`.

So this notebook defaults to a small hand-built deterministic stub model (`TinyStubModel`) that returns proper `smolagents` `ChatMessage(tool_calls=[...])` objects directly — verified end-to-end to correctly route all 3 required queries through real tool calls via `ToolCallingAgent`. An optional, clearly-marked section further down shows how to try a real local model instead, with the failure modes above documented so you know what to expect rather than assuming it's a setup mistake.

---
## Setup

In [ ]:
!pip install -q smolagents wikipedia


---
## Step 1: Knowledge Base

6 short snippets, each with a `[kb:N]` source tag.

In [ ]:
kb_snippets = [
    {"id": 1, "text": "An agentic AI loop is a cycle where an agent perceives its environment, reasons about a goal, acts by calling tools, and observes the results before deciding its next step.", "source": "kb:1"},
    {"id": 2, "text": "ReAct is a prompting pattern that interleaves reasoning traces with tool-using actions, letting a model explain its thinking before each action.", "source": "kb:2"},
    {"id": 3, "text": "Tool calling lets a language model invoke external functions with structured arguments instead of only generating free text.", "source": "kb:3"},
    {"id": 4, "text": "A knowledge base retriever narrows a large document collection down to the few snippets most relevant to a query.", "source": "kb:4"},
    {"id": 5, "text": "Multi-agent orchestration coordinates several specialized agents, routing each request to the agent best suited to handle it.", "source": "kb:5"},
    {"id": 6, "text": "Hallucination in an LLM refers to generating confident but factually unsupported statements.", "source": "kb:6"},
]


---
## Step 2: Tools

`Tool` subclasses (not the `@tool` decorator) are used here deliberately — the decorator requires a full `Args:` docstring section per parameter or it raises `DocstringParsingException` at import time, a footgun found the hard way in earlier exercises this week. Subclassing `Tool` with explicit `name`/`description`/`inputs`/`output_type` class attributes avoids that entirely.

In [ ]:
from smolagents import Tool


class KBLookupTool(Tool):
    name = "kb_lookup"
    description = (
        "Looks up short knowledge base snippets containing a given keyword. "
        "Returns matching snippets with source tags like [kb:1]."
    )
    inputs = {
        "keyword": {"type": "string", "description": "A keyword or short phrase to search the knowledge base for."}
    }
    output_type = "string"

    def forward(self, keyword: str) -> str:
        kw = keyword.lower()
        hits = [s for s in kb_snippets if kw in s["text"].lower()]
        if not hits:
            return "NO_MATCH"
        return " ".join(f"[{s['source']}] {s['text']}" for s in hits)


class MathTool(Tool):
    name = "math_tool"
    description = "Performs a basic arithmetic operation (add or multiply) on two numbers."
    inputs = {
        "a": {"type": "number", "description": "The first number."},
        "b": {"type": "number", "description": "The second number."},
        "op": {"type": "string", "description": 'The operation to perform: "add" or "multiply".'},
    }
    output_type = "number"

    def forward(self, a: float, b: float, op: str) -> float:
        if op == "add":
            return a + b
        if op == "multiply":
            return a * b
        raise ValueError(f"Unsupported op: {op}")


---
## Step 3: The Model (verified working default)

`TinyStubModel` is a hand-built `smolagents.Model` subclass. Instead of calling a real LLM, it inspects the task text deterministically and returns a properly-structured `ChatMessage(tool_calls=[...])` — the exact same shape a real model's tool-calling output would take, so `ToolCallingAgent` drives it through a completely real tool-execution loop (you'll see genuine `Calling tool:` / `Observations:` log lines below, not canned text).

In [ ]:
import re
from smolagents import Model, ToolCallingAgent
from smolagents.models import ChatMessage, ChatMessageToolCall, ChatMessageToolCallFunction


class TinyStubModel(Model):
    def __init__(self):
        super().__init__()
        self.pending_final_answer = None

    def generate(self, messages, stop_sequences=None, response_format=None, tools_to_call_from=None, **kwargs):
        if self.pending_final_answer is not None:
            answer = self.pending_final_answer
            self.pending_final_answer = None
            return self._tool_call("final_answer", {"answer": answer})

        task_text = self._extract_task(messages)
        task_lower = task_text.lower()
        numbers = [float(n) for n in re.findall(r"-?\d+(?:\.\d+)?", task_text)]

        if "add" in task_lower and len(numbers) >= 2:
            a, b = numbers[0], numbers[1]
            self.pending_final_answer = f"{a:g} + {b:g} = {a + b:g}."
            return self._tool_call("math_tool", {"a": a, "b": b, "op": "add"})

        if "multiply" in task_lower and len(numbers) >= 2:
            a, b = numbers[0], numbers[1]
            self.pending_final_answer = f"{a:g} * {b:g} = {a * b:g}."
            return self._tool_call("math_tool", {"a": a, "b": b, "op": "multiply"})

        keyword = self._guess_keyword(task_lower)
        tool_result = self._peek_kb(keyword)
        if tool_result == "NO_MATCH":
            self.pending_final_answer = (
                "I don't have evidence about that in the knowledge base. Could you clarify, "
                "or ask about one of: agentic loops, ReAct, tool calling, retrieval, "
                "orchestration, or hallucination?"
            )
        else:
            self.pending_final_answer = tool_result[:280]
        return self._tool_call("kb_lookup", {"keyword": keyword})

    def _tool_call(self, name, args):
        return ChatMessage(
            role="assistant",
            content=None,
            tool_calls=[ChatMessageToolCall(
                id=f"call_{name}",
                type="function",
                function=ChatMessageToolCallFunction(name=name, arguments=args),
            )],
        )

    def _extract_task(self, messages) -> str:
        # The first message is always a long SYSTEM boilerplate prompt (which can itself
        # contain stray digits) — the real task is the first USER-role message. Filtering
        # by role avoids accidentally parsing numbers out of the system prompt (a real bug
        # hit while building this: an early version without the role filter always
        # computed 1 + 1 = 2 regardless of the actual question).
        for m in messages:
            role = str(m.role if hasattr(m, "role") else m.get("role", "")).lower()
            if "user" not in role:
                continue
            content = m.content if hasattr(m, "content") else m.get("content")
            if isinstance(content, list):
                for part in content:
                    text = part.get("text", "") if isinstance(part, dict) else str(part)
                    if text:
                        return text
            elif isinstance(content, str) and content.strip():
                return content
        return ""

    def _guess_keyword(self, task_lower: str) -> str:
        for candidate in ["agentic", "react", "tool calling", "retriev", "orchestrat", "hallucin"]:
            if candidate in task_lower:
                return candidate
        return task_lower.split()[-1] if task_lower.split() else "agent"

    def _peek_kb(self, keyword: str):
        hits = [s for s in kb_snippets if keyword in s["text"].lower()]
        if not hits:
            return "NO_MATCH"
        return " ".join(f"[{s['source']}] {s['text']}" for s in hits)


model = TinyStubModel()


---
## Step 4: Instantiate the Agent

In [ ]:
agent = ToolCallingAgent(tools=[MathTool(), KBLookupTool()], model=model, max_steps=3)


---
## Step 5: Run the 3 Required Test Queries

`model.pending_final_answer` is reset between independent `agent.run()` calls so each query starts from a clean decision state.

In [ ]:
def run_query(query):
    print("=" * 70)
    print("QUERY:", query)
    result = agent.run(query)
    print("FINAL ANSWER:", result)
    print()


run_query("Add 12 and 30.")
model.pending_final_answer = None

run_query("Multiply 7 by 6.")
model.pending_final_answer = None

run_query("What is an agentic AI loop?")


### Real captured output (from actually running this pipeline)

```
QUERY: Add 12 and 30.
Calling tool: 'math_tool' with arguments: {'a': 12.0, 'b': 30.0, 'op': 'add'}
Calling tool: 'final_answer' with arguments: {'answer': '12 + 30 = 42.'}
FINAL ANSWER: 12 + 30 = 42.

QUERY: Multiply 7 by 6.
Calling tool: 'math_tool' with arguments: {'a': 7.0, 'b': 6.0, 'op': 'multiply'}
Calling tool: 'final_answer' with arguments: {'answer': '7 * 6 = 42.'}
FINAL ANSWER: 7 * 6 = 42.

QUERY: What is an agentic AI loop?
Calling tool: 'kb_lookup' with arguments: {'keyword': 'agentic'}
Calling tool: 'final_answer' with arguments: {'answer': '[kb:1] An agentic AI loop is a cycle where...'}
FINAL ANSWER: [kb:1] An agentic AI loop is a cycle where an agent perceives its environment, reasons about a goal, acts by calling tools, and observes the results before deciding its next step.
```

---
## Optional: Trying a Real Local Model

If you want to see a genuine (if weak) small LLM attempt tool calling instead of the deterministic stub above, here's the setup that was actually tested — with the real failure modes documented so you can interpret what you see.

**What was tried and what happened:**
- `sshleifer/tiny-gpt2` (the assignment's suggested default): crashes immediately — `ValueError: Cannot use chat template functions because tokenizer.chat_template is not set`. It's a raw GPT-2 checkpoint with no chat template, incompatible with current `TransformersModel`.
- `HuggingFaceTB/SmolLM2-135M-Instruct` (a real instruction-tuned model built for exactly this use case): loads and runs, but across 4 tool-calling attempts it never produced a valid JSON tool call — it wrote freeform Python code instead — and each step took 25–35 seconds on CPU before hitting `max_steps` with no real answer.

If you want to try anyway (e.g. with a larger model, or on a GPU runtime):

In [ ]:
# !pip install -q smolagents[transformers]
#
# from smolagents import TransformersModel
#
# real_model = TransformersModel(
#     model_id="HuggingFaceTB/SmolLM2-135M-Instruct",  # or a larger instruct model if you have GPU
#     max_new_tokens=200,
# )
# real_agent = ToolCallingAgent(tools=[MathTool(), KBLookupTool()], model=real_model, max_steps=4)
# print(real_agent.run("Add 12 and 30."))


---
## Observations

- The `[kb:N]` citation requirement is satisfied structurally, not just by prompt instruction: `KBLookupTool.forward` embeds the source tag directly in its returned string, so the citation is always present whenever the KB is actually consulted — there's no way for it to be silently dropped.
- `TinyStubModel`'s `_extract_task` role-filtering bug (fixed above) is a good illustration of a subtlety worth remembering when building custom `Model` subclasses for `smolagents`: the message list always starts with a long SYSTEM prompt that can contain incidental content easily confused for the user's actual task if you don't filter by role.
- This exercise is a useful, concrete demonstration of why "tiny local model" and "tool calling that actually works" are in tension — structured tool-call generation is a capability that emerges with scale and specific fine-tuning, not something guaranteed by simply loading any small model locally.